# Global Popularity baseline

This notebook inspects the immutable task-02 artifact. Score computation, history-only loading, seen filtering, model selection, and canonical evaluation live in `popularity.py` and `scripts/run_global_popularity.py`.

In [ ]:
import json
from pathlib import Path

import polars as pl

from popularity import (
    ITEM_RANKING_SCHEMA,
    GlobalPopularityModel,
    PopularityDataLoader,
    PopularityScore,
)
from interfaces import FINAL_RECOMMENDATION_SCHEMA

pl.Config.set_tbl_rows(30)

In [ ]:
artifact_dir = Path("artifacts/task02_global_popularity_v1")
config = json.loads((artifact_dir / "config.json").read_text())
metrics = json.loads((artifact_dir / "metrics.json").read_text())
config["selection"], metrics["selected_score_type"]

In [ ]:
selection_rows = []
for score_type, values in metrics["selection_summary"].items():
    selection_rows.append({"score_type": score_type, **values})
pl.DataFrame(selection_rows).sort(
    "mean_precision_at_20_all_targets", descending=True
)

In [ ]:
pl.DataFrame(
    [
        {
            "fold_cutoff": fold["fold"]["cutoff"],
            **score,
        }
        for fold in metrics["selection_folds"]
        for score in fold["scores"]
    ]
).select(
    "fold_cutoff",
    "score_type",
    "precision_at_20_all_targets",
    "precision_at_20_labeled_users",
    "candidate_recall",
    "candidate_oracle_p20_all_targets",
)

In [ ]:
canonical_keys = [
    "precision_at_20_all_targets",
    "precision_at_20_labeled_users",
    "candidate_recall",
    "candidate_user_hit_rate",
    "candidate_oracle_p20_all_targets",
    "candidate_oracle_p20_labeled_users",
    "coverage",
    "mean_candidate_count",
    "final_hits",
]
pl.DataFrame(
    {
        "metric": canonical_keys,
        "value": [metrics["canonical"][key] for key in canonical_keys],
    }
)

In [ ]:
item_ranking = pl.read_parquet(artifact_dir / "item_ranking.parquet")
recommendations = pl.read_parquet(artifact_dir / "recommendations.parquet")
assert item_ranking.schema == ITEM_RANKING_SCHEMA
assert recommendations.schema == FINAL_RECOMMENDATION_SCHEMA
assert metrics["deterministic_recommendations_match"]
item_ranking.head(20), recommendations.head()

## Interpretation

The selected score is determined only by the three earlier rolling folds. The canonical section contains exactly one score configuration and is the comparable task-02 baseline. Candidate diagnostics use 200 unseen items per user; Precision@20 uses the first 20.